# ETAPA: Retomada R2

Este notebook executa o diagnostico RAW por dominio (R2): leitura, estrutura, cobertura temporal e auditoria de parsing, gerando artefatos objetivos para o Relatorio RAW.

Regras:
- Somente leitura dos dados originais.
- Nao publicar/migrar para MinIO.
- Saidas em outputs/R2.

In [ ]:
from __future__ import annotations

import os, sys, json, io, warnings, csv
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm

pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 220)

BASE_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico')
MANIFEST_PATH = Path('/home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv')
RAW_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais')
OUT_DIR = Path('/home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime('%Y-%m-%d_%H%M%S')

print('Python:', sys.version)
print('Pandas:', pd.__version__)
print('BASE_DIR:', BASE_DIR)
print('MANIFEST_PATH:', MANIFEST_PATH)
print('RAW_DIR:', RAW_DIR)
print('OUT_DIR:', OUT_DIR)
print('RUN_TS:', RUN_TS)


Python: 3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:46:25) [GCC 14.3.0]
Pandas: 2.3.3
BASE_DIR: /home/wilson/Maringa/fase_1_diagnostico
MANIFEST_PATH: /home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv
RAW_DIR: /home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais
OUT_DIR: /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2
RUN_TS: 2026-01-05_173712


# ETAPA: Carregar manifesto e construir indice de arquivos

In [14]:
dfm = pd.read_csv(MANIFEST_PATH)
print('dfm.shape:', dfm.shape)
print('dfm.columns:', list(dfm.columns))
display(dfm.head(20))

PATH_COL = 'path_local'
if PATH_COL not in dfm.columns:
    raise RuntimeError(f'Coluna esperada ausente no manifesto: {PATH_COL}')

records = []
raw_root = RAW_DIR.resolve()

for raw_path in tqdm(dfm[PATH_COL].astype(str), desc='Normalizando paths (relativo ao RAW_DIR)'):
    p = Path(raw_path)
    if not p.is_absolute():
        p = (BASE_DIR / p).resolve()
    else:
        p = p.resolve()

    try:
        rel = p.relative_to(raw_root).as_posix()
        in_raw = True
    except Exception:
        rel = ''
        in_raw = False

    if rel:
        dom = rel.split('/')[0] if '/' in rel else rel
    else:
        dom = 'fora_raw'

    try:
        size_bytes = p.stat().st_size
    except Exception:
        size_bytes = None

    records.append({
        'path_local': str(p),
        'rel_norm': rel,
        'dominio': dom,
        'suffix': p.suffix.lower(),
        'size_bytes': size_bytes,
        'in_raw_dir': in_raw
    })


df_idx = pd.DataFrame(records)

print('df_idx.shape:', df_idx.shape)
display(df_idx.head(20))

df_idx.to_csv(OUT_DIR / 'r2_file_index.csv', index=False)
print('Salvo:', OUT_DIR / 'r2_file_index.csv')


dfm.shape: (145, 24)
dfm.columns: ['id_arquivo', 'path_local', 'pasta_raiz', 'pasta_raiz_canonica', 'nome_arquivo', 'extensao', 'tamanho_bytes', 'data_modificacao', 'hash_sha256', 'dominio', 'forno', 'granularidade', 'origem_fisica', 'bucket_minio', 'path_minio', 'status_pipeline', 'fonte_manifesto', 'observacoes_dominio', 'observacoes_forno', 'observacoes_granularidade', 'observacoes_pipeline', 'fontes_manifesto', 'qtd_arquivos_reportada_manifesto', 'descricao_manifesto']


,id_arquivo,path_local,pasta_raiz,pasta_raiz_canonica,nome_arquivo,extensao,tamanho_bytes,data_modificacao,hash_sha256,dominio,forno,granularidade,origem_fisica,bucket_minio,path_minio,status_pipeline,fonte_manifesto,observacoes_dominio,observacoes_forno,observacoes_granularidade,observacoes_pipeline,fontes_manifesto,qtd_arquivos_reportada_manifesto,descricao_manifesto
0,0000b88858fe4a0371b224eb49a4f47f544c709322f38e...,dados/dados_iniciais/Consumo Fornos/2019_F1_Co...,Consumo Fornos,Consumo Fornos,2019_F1_Consumo.csv,.csv,42566405,2025-04-29T15:23:46+00:00,e2d6122f72312561518cd3dfbcfc425ca210fa1140a2e1...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/01/2019_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
1,e684247a48fd1fbc45f967124314b326cbc886b004fac0...,dados/dados_iniciais/Consumo Fornos/2020_F1_Co...,Consumo Fornos,Consumo Fornos,2020_F1_Consumo.csv,.csv,28785370,2025-04-29T15:25:02+00:00,e2f659eb79b9efb6d1a1aedf577b2f5ee9ce0b38cc8309...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2020_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
2,d2243389167f2f277659ebe70c82160fefd9ad15680e62...,dados/dados_iniciais/Consumo Fornos/2021_F1_Co...,Consumo Fornos,Consumo Fornos,2021_F1_Consumo.csv,.csv,29619879,2025-04-29T15:26:22+00:00,bca0695b76a186d6e603ddc6775d2379d4d094d2f66f89...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2021_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
3,df3919b433b24e293cb9a2fa9bb21e045dae7dffd169ff...,dados/dados_iniciais/Consumo Fornos/2022_F1_Co...,Consumo Fornos,Consumo Fornos,2022_F1_Consumo.csv,.csv,29118391,2025-04-29T15:27:40+00:00,7bd095e671301d332f6131d223a2343055f0f540e7e903...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2022_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
4,d31c7691b4c1d040fa135109d02b2fb0d0497587a1b309...,dados/dados_iniciais/Consumo Fornos/2023_F1_Co...,Consumo Fornos,Consumo Fornos,2023_F1_Consumo.csv,.csv,28111195,2025-04-29T15:28:52+00:00,600099b5cf736d738b126a60102b448129c63008357c31...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2023_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
5,23d9f0e4a454a7c4facf26bbbafb4be358f17b996a5924...,dados/dados_iniciais/Consumo Fornos/2024_F1_Co...,Consumo Fornos,Consumo Fornos,2024_F1_Consumo.csv,.csv,27148206,2025-04-29T15:30:04+00:00,488d5d081b8e32f72616373c97149e98f45c3ec71a83ba...,consumo_fornos,F1,nao_definida,local,maringa-raw,consumo_fornos/F1/nao_definida/0000/02/2024_F1...,raw_local,"index_resumo_por_pasta,manifest_index",NaN,NaN,Sem indicacao clara para consumo_fornos,path_minio com ano/mes placeholder; granularid...,"index_resumo_por_pasta,manifest_index",78,"2018_CONSUMO; F5, F4, F3, F2; CSV; 2019_CONSUM..."
6,d4635e66e027b7a960fc6fff149f5c3cb07b052c4a5409...,dados/dados_iniciais/Consumo Fornos/2025_F1_Co...,Consumo Fornos,Consumo

Normalizando paths (relativo ao RAW_DIR): 100%|██████████| 145/145 [00:00<00:00, 15475.17it/s]

df_idx.shape: (145, 6)


,path_local,rel_norm,dominio,suffix,size_bytes,in_raw_dir
0,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F1_Consumo.csv,Consumo Fornos,.csv,42566405,True
1,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F1_Consumo.csv,Consumo Fornos,.csv,28785370,True
2,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2021_F1_Consumo.csv,Consumo Fornos,.csv,29619879,True
3,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2022_F1_Consumo.csv,Consumo Fornos,.csv,29118391,True
4,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2023_F1_Consumo.csv,Consumo Fornos,.csv,28111195,True
5,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2024_F1_Consumo.csv,Consumo Fornos,.csv,27148206,True
6,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2025_F1_Consumo.csv,Consumo Fornos,.csv,25851580,True
7,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2018_F2_Consumo.csv,Consumo Fornos,.csv,28950526,True
8,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F2_Consumo.csv,Consumo Fornos,.csv,28760963,True
9,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F2_Consumo.csv,Consumo Fornos,.csv,29180244,True


Salvo: /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_file_index.csv


# ETAPA: Auditoria de parsing (CSV) e extracao de metadados basicos

In [ ]:
def sniff_delimiter(sample_text: str) -> str:
    # heuristica simples: escolhe o separador mais provavel
    candidates = [',',';','	','|']
    counts = {c: sample_text.count(c) for c in candidates}
    best = max(counts, key=counts.get)
    return best

parse_rows = []

csv_rows = df_idx[(df_idx['suffix'] == '.csv') & (df_idx['in_raw_dir'])].copy()
print('CSV files (no RAW_DIR):', csv_rows.shape[0])

raw_root = RAW_DIR.resolve()
encodings_try = ['utf-8', 'latin-1', 'cp1252']

for _, row in tqdm(csv_rows.iterrows(), total=csv_rows.shape[0], desc='Auditando CSVs'):
    p = Path(row['path_local']).resolve()
    info = {
        'path_local': str(p),
        'rel_norm': row['rel_norm'],
        'dominio': row['dominio'],
        'suffix': row['suffix'],
        'ok': False,
        'encoding_used': None,
        'delimiter': None,
        'n_rows': None,
        'n_cols': None,
        'columns': None,
        'date_cols_guess': None,
        'min_date': None,
        'max_date': None,
        'error': None
    }

    try:
        p.relative_to(raw_root)
    except Exception:
        info['error'] = 'fora_RAW_DIR'
        parse_rows.append(info)
        continue

    if not p.exists():
        info['error'] = 'arquivo_inexistente'
        parse_rows.append(info)
        continue

    try:
        df = None
        last_err = None
        for enc in encodings_try:
            try:
                text_content = p.read_text(encoding=enc, errors='replace')
                sample = text_content[:5000]
                delim = sniff_delimiter(sample)
                info['delimiter'] = delim

                read_kwargs = {'sep': delim, 'engine': 'python'}
                if row['dominio'] == 'Supervisorio Forno 4':
                    read_kwargs['quoting'] = csv.QUOTE_NONE

                df = pd.read_csv(io.StringIO(text_content), **read_kwargs)
                info['encoding_used'] = enc
                break
            except Exception as inner_e:
                last_err = inner_e
                continue

        if df is None:
            info['error'] = repr(last_err)
            parse_rows.append(info)
            continue

        info['ok'] = True
        info['n_rows'] = int(df.shape[0])
        info['n_cols'] = int(df.shape[1])
        info['columns'] = '|'.join([str(c) for c in df.columns.tolist()])

        # tentativa simples de detectar colunas temporais
        date_candidates = [c for c in df.columns if any(k in str(c).lower() for k in ['data','date','dt','hora','time'])]
        info['date_cols_guess'] = '|'.join([str(c) for c in date_candidates]) if date_candidates else ''

        # se houver candidata, tentar parse e obter min/max
        if date_candidates:
            c0 = date_candidates[0]
            series = df[c0]
            if series.dtype == object:
                series = series.astype(str).str.strip('"')
            s = pd.to_datetime(series, errors='coerce', dayfirst=False, utc=True)
            s = s.dropna()
            if len(s) > 0:
                info['min_date'] = str(s.min())
                info['max_date'] = str(s.max())

    except Exception as e:
        info['error'] = repr(e)

    parse_rows.append(info)


df_parse = pd.DataFrame(parse_rows)
print('df_parse.shape:', df_parse.shape)
display(df_parse.head(20))

df_parse.to_csv(OUT_DIR / 'r2_parse_audit.csv', index=False)
print('Salvo:', OUT_DIR / 'r2_parse_audit.csv')


CSV files (no RAW_DIR): 124


Auditando CSVs:  61%|██████▏   | 76/124 [01:33<00:02, 16.96it/s]/tmp/ipykernel_21407/1382332066.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(series, errors='coerce', dayfirst=False, utc=True)
Auditando CSVs:  98%|█████████▊| 122/124 [02:02<00:01,  1.20it/s]/tmp/ipykernel_21407/1382332066.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(series, errors='coerce', dayfirst=False, utc=True)
Auditando CSVs:  99%|█████████▉| 123/124 [02:03<00:00,  1.17it/s]/tmp/ipykernel_21407/1382332066.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  s = pd.to_date

df_parse.shape: (124, 14)


,path_local,rel_norm,dominio,suffix,ok,encoding_used,delimiter,n_rows,n_cols,columns,date_cols_guess,min_date,max_date,error
0,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",961515,42,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2019-01-01 00:00:00+00:00,2019-12-31 00:00:00+00:00,None
1,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019752,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2020-01-01 00:00:00+00:00,2020-12-31 00:00:00+00:00,None
2,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2021_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1012132,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2021-01-01 00:00:00+00:00,2021-12-31 00:00:00+00:00,None
3,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2022_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",989761,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2022-01-01 00:00:00+00:00,2022-12-31 00:00:00+00:00,None
4,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2023_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",959436,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2023-01-01 00:00:00+00:00,2023-12-31 00:00:00+00:00,None
5,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2024_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",925925,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2024-01-01 00:00:00+00:00,2024-12-31 00:00:00+00:00,None
6,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2025_F1_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",907889,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2025-01-01 00:00:00+00:00,2025-04-28 00:00:00+00:00,None
7,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2018_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019780,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2018-01-01 00:00:00+00:00,2018-12-31 00:00:00+00:00,None
8,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2019_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",1019762,26,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2019-01-01 00:00:00+00:00,2019-12-31 00:00:00+00:00,None
9,/home/wilson/Maringa/fase_1_diagnostico/dados/...,Consumo Fornos/2020_F2_Consumo.csv,Consumo Fornos,.csv,True,utf-8,",",997916,27,Forno|Data|Folha|Bal.|Silo|C�digo|Descri��o|Um...,Data,2020-01-01 00:00:00+00:00,2020-12-31 00:00:00+00:00,None


Salvo: /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_parse_audit.csv


# ETAPA: Sumarios por dominio e cobertura temporal

In [16]:
# Sumario por dominio
agg_domain = df_parse.groupby('dominio', dropna=False).agg(
    n_files=('path_local','count'),
    n_ok=('ok','sum'),
    n_fail=('ok', lambda s: int((~s).sum())),
    total_rows=('n_rows', lambda s: int(pd.to_numeric(s, errors='coerce').fillna(0).sum())),
    total_cols_mean=('n_cols', lambda s: float(pd.to_numeric(s, errors='coerce').dropna().mean()) if pd.to_numeric(s, errors='coerce').dropna().shape[0] else np.nan)
).reset_index()

print('agg_domain.shape:', agg_domain.shape)
display(agg_domain.head(20))
agg_domain.to_csv(OUT_DIR / 'r2_domain_summary.csv', index=False)

# Cobertura temporal por dominio (quando houver min/max)
df_cov = df_parse[df_parse['min_date'].notna() & df_parse['max_date'].notna()].copy()
if df_cov.shape[0] > 0:
    cov_domain = df_cov.groupby('dominio', dropna=False).agg(
        min_date=('min_date', 'min'),
        max_date=('max_date', 'max'),
        n_files_with_date=('path_local','count')
    ).reset_index()
else:
    cov_domain = pd.DataFrame(columns=['dominio','min_date','max_date','n_files_with_date'])

print('cov_domain.shape:', cov_domain.shape)
display(cov_domain.head(20))

cov_domain.to_csv(OUT_DIR / 'r2_date_coverage.csv', index=False)

# placeholder de forno_summary: somente se houver coluna identificavel (ex.: F1/F2 no path)
# regra: extrair forno do rel_norm se contiver '/F1/' ou 'F1_' etc.
import re

def extract_forno(rel: str) -> str:
    if not isinstance(rel, str):
        return ''
    m = re.search(r'\bF[1-9]\b', rel)
    if m:
        return m.group(0)
    m2 = re.search(r'(F[1-9])_', rel)
    if m2:
        return m2.group(1)
    return ''


df_parse['forno'] = [extract_forno(r) for r in df_parse['rel_norm'].astype(str)]
forno_summary = df_parse.groupby(['dominio','forno'], dropna=False).agg(
    n_files=('path_local','count'),
    n_ok=('ok','sum'),
    n_fail=('ok', lambda s: int((~s).sum()))
).reset_index()

print('forno_summary.shape:', forno_summary.shape)
display(forno_summary.head(20))

forno_summary.to_csv(OUT_DIR / 'r2_forno_summary.csv', index=False)


agg_domain.shape: (6, 6)


,dominio,n_files,n_ok,n_fail,total_rows,total_cols_mean
0,Consumo Fornos,39,39,0,38123141,27.410256
1,Corridas,40,40,0,114949,34.075000
2,Eletrodo,1,1,0,761,5.000000
3,Informações Diária,40,40,0,14232,47.225000
4,Supervisorio Forno 4,2,2,0,486250,228.000000
5,Supervisorio Forno 5,2,2,0,242099,25.500000


cov_domain.shape: (6, 4)


,dominio,min_date,max_date,n_files_with_date
0,Consumo Fornos,2018-01-01 00:00:00+00:00,2025-04-28 00:00:00+00:00,39
1,Corridas,2018-01-01 00:00:00+00:00,2025-04-29 00:00:00+00:00,40
2,Eletrodo,2021-01-02 00:00:00+00:00,2025-12-03 00:00:00+00:00,1
3,Informações Diária,2018-01-01 00:00:00+00:00,2025-04-29 00:00:00+00:00,40
4,Supervisorio Forno 4,2023-09-29 04:00:00+00:00,2025-01-02 03:58:00+00:00,2
5,Supervisorio Forno 5,2021-01-06 00:00:00+00:00,2024-12-05 23:00:00+00:00,2


forno_summary.shape: (19, 5)


,dominio,forno,n_files,n_ok,n_fail
0,Consumo Fornos,F1,7,7,0
1,Consumo Fornos,F2,8,8,0
2,Consumo Fornos,F3,8,8,0
3,Consumo Fornos,F4,8,8,0
4,Consumo Fornos,F5,8,8,0
5,Corridas,F1,8,8,0
6,Corridas,F2,8,8,0
7,Corridas,F3,8,8,0
8,Corridas,F4,8,8,0
9,Corridas,F5,8,8,0


# ETAPA: Gerar sumario e relatorio R2

In [ ]:
summary = {
    'run_ts': RUN_TS,
    'manifest_path': str(MANIFEST_PATH),
    'dados_iniciais_dir': str(RAW_DIR),
    'n_total_files_manifest': int(df_idx.shape[0]),
    'n_csv_files': int((df_idx['suffix'] == '.csv').sum()),
    'n_csv_ok': int(df_parse['ok'].sum()),
    'n_csv_fail': int((~df_parse['ok']).sum()),
    'n_dominios': int(agg_domain.shape[0])
}

report = []
report.append('# Retomada R2 - Relatorio de Diagnostico RAW')
report.append('')
report.append(f'run_ts: {RUN_TS}')
report.append(f'manifest_path: {MANIFEST_PATH}')
report.append(f'dados_iniciais_dir: {RAW_DIR}')
report.append('')
report.append('## Sumario')
for k, v in summary.items():
    if k in ['run_ts','manifest_path','dados_iniciais_dir']:
        continue
    report.append(f'- {k}: {v}')

report.append('')
report.append('## Artefatos gerados')
for f in ['r2_file_index.csv','r2_parse_audit.csv','r2_domain_summary.csv','r2_forno_summary.csv','r2_date_coverage.csv','r2_summary.json']:
    report.append(f'- {f}')

report_content = '\n'.join(report)

(Path(OUT_DIR) / 'r2_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(Path(OUT_DIR) / 'r2_report.md').write_text(report_content, encoding='utf-8')

print('Salvos:')
for f in ['r2_file_index.csv','r2_parse_audit.csv','r2_domain_summary.csv','r2_forno_summary.csv','r2_date_coverage.csv','r2_summary.json','r2_report.md']:
    print('-', OUT_DIR / f)

print('\nResumo:')
print(json.dumps(summary, ensure_ascii=False, indent=2))


Salvos:
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_file_index.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_parse_audit.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_domain_summary.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_forno_summary.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_date_coverage.csv
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_summary.json
- /home/wilson/Maringa/fase_1_diagnostico/3ª-Retomada/outputs/R2/r2_report.md

Resumo:
{
  "run_ts": "2026-01-05_173712",
  "manifest_path": "/home/wilson/Maringa/fase_1_diagnostico/dados/manifestos_final/manifest_unificado.csv",
  "dados_iniciais_dir": "/home/wilson/Maringa/fase_1_diagnostico/dados/dados_iniciais",
  "n_total_files_manifest": 145,
  "n_csv_files": 124,
  "n_csv_ok": 124,
  "n_csv_fail": 0,
  "n_dominios": 6
}
